In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [3]:
from xgboost import XGBRegressor

In [4]:
df = pd.read_csv("../data/processed/final_features.csv")

print(df.shape)
df.head()

(10000, 33)


,year,state,district,season,crop_type,seed_variety,area_sown_hectares,irrigation_type,rainfall_mm,temperature_min_c,...,yield_tonnes_per_hectare,sowing_year,sowing_month,sowing_day,sowing_day_of_year,temperature_range_c,rainfall_per_day,total_npk_kg_ha,np_ratio,kn_ratio
0,2022,Karnataka,Belagavi,Rabi,Maize,HQPM-1,1.52,Rainfed,573.7,17.5,...,2.82,2022,11,10,314,19.5,5.794949,221.0,1.541985,1.086634
1,2019,Karnataka,Kalaburagi,Rabi,Maize,HQPM-1,22.19,Rainfed,805.9,17.3,...,2.33,2019,11,25,329,14.2,7.393578,300.3,2.497600,0.523382
2,2019,Punjab,Amritsar,Rabi,Wheat,HD-2967,8.50,Drip Irrigated,343.2,7.5,...,2.31,2019,11,15,319,18.7,2.908475,139.8,1.144231,0.475630
3,2017,Madhya Pradesh,Jabalpur,Rabi,Wheat,HD-3086,7.48,Irrigated,339.7,14.9,...,1.72,2017,10,29,302,12.3,2.342759,227.1,1.935323,0.429306
4,2022,Rajasthan,Bikaner,Kharif,Cotton,Bunny-Bt,6.44,Rainfed,564.1,20.3,...,1.46,2022,6,4,155,13.5,3.016578,200.6,5.274900,0.325529


In [5]:
df.columns.tolist()

['year',
 'state',
 'district',
 'season',
 'crop_type',
 'seed_variety',
 'area_sown_hectares',
 'irrigation_type',
 'rainfall_mm',
 'temperature_min_c',
 'temperature_max_c',
 'temperature_avg_c',
 'humidity_pct',
 'growing_season_days',
 'soil_ph',
 'nitrogen_kg_ha',
 'phosphorus_kg_ha',
 'potassium_kg_ha',
 'soil_type',
 'soil_moisture_pct',
 'previous_yield_tonnes_ha',
 'yield_trend_pct_yoy',
 'ndvi',
 'yield_tonnes_per_hectare',
 'sowing_year',
 'sowing_month',
 'sowing_day',
 'sowing_day_of_year',
 'temperature_range_c',
 'rainfall_per_day',
 'total_npk_kg_ha',
 'np_ratio',
 'kn_ratio']

In [6]:
df.isnull().sum()

year                        0
state                       0
district                    0
season                      0
crop_type                   0
seed_variety                0
area_sown_hectares          0
irrigation_type             0
rainfall_mm                 0
temperature_min_c           0
temperature_max_c           0
temperature_avg_c           0
humidity_pct                0
growing_season_days         0
soil_ph                     0
nitrogen_kg_ha              0
phosphorus_kg_ha            0
potassium_kg_ha             0
soil_type                   0
soil_moisture_pct           0
previous_yield_tonnes_ha    0
yield_trend_pct_yoy         0
ndvi                        0
yield_tonnes_per_hectare    0
sowing_year                 0
sowing_month                0
sowing_day                  0
sowing_day_of_year          0
temperature_range_c         0
rainfall_per_day            0
total_npk_kg_ha             0
np_ratio                    0
kn_ratio                    0
dtype: int

In [7]:
TARGET = "yield_tonnes_per_hectare"

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [9]:
print("Features:", X.shape)
print("Target:", y.shape)

Features: (10000, 32)
Target: (10000,)


In [10]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['year', 'area_sown_hectares', 'rainfall_mm', 'temperature_min_c', 'temperature_max_c', 'temperature_avg_c', 'humidity_pct', 'growing_season_days', 'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha', 'soil_moisture_pct', 'previous_yield_tonnes_ha', 'yield_trend_pct_yoy', 'ndvi', 'sowing_year', 'sowing_month', 'sowing_day', 'sowing_day_of_year', 'temperature_range_c', 'rainfall_per_day', 'total_npk_kg_ha', 'np_ratio', 'kn_ratio']

Categorical Features:
['state', 'district', 'season', 'crop_type', 'seed_variety', 'irrigation_type', 'soil_type']


C:\Users\HP\AppData\Local\Temp\ipykernel_16476\3566523423.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 8000
Testing samples: 2000


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [13]:
gradient_boosting = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

In [14]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", gradient_boosting)
    ]
)

In [15]:
gb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['year','state','district',...,'total_npk_kg_ha','np_ratio','kn_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columns

In [16]:
gb_predictions = gb_pipeline.predict(X_test)

In [17]:
gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_predictions))
gb_r2 = r2_score(y_test, gb_predictions)

print("Gradient Boosting Results")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R²:", gb_r2)

Gradient Boosting Results
MAE: 0.19894214326427856
RMSE: 0.27076848876496634
R²: 0.8923367039562979


In [18]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

In [19]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [20]:
xgb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['year','state','district',...,'total_npk_kg_ha','np_ratio','kn_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columns

In [21]:
xgb_predictions = xgb_pipeline.predict(X_test)

In [22]:
xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))
xgb_r2 = r2_score(y_test, xgb_predictions)

print("XGBoost Results")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R²:", xgb_r2)

XGBoost Results
MAE: 0.1402165996569395
RMSE: 0.18767848430989975
R²: 0.9482750113439801


In [23]:
svr_model = SVR(
    kernel="rbf",
    C=100,
    gamma="scale",
    epsilon=0.1
)

In [24]:
svr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", svr_model)
    ]
)

In [25]:
svr_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['year','state','district',...,'total_npk_kg_ha','np_ratio','kn_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columns

In [26]:
svr_predictions = svr_pipeline.predict(X_test)

In [27]:
svr_mae = mean_absolute_error(y_test, svr_predictions)
svr_rmse = np.sqrt(mean_squared_error(y_test, svr_predictions))
svr_r2 = r2_score(y_test, svr_predictions)

print("SVR Results")
print("MAE:", svr_mae)
print("RMSE:", svr_rmse)
print("R²:", svr_r2)

SVR Results
MAE: 0.17318839777362155
RMSE: 0.22824924531623111
R²: 0.9234949453439123


In [28]:
results = pd.DataFrame({
    "Model": [
        "Gradient Boosting",
        "XGBoost",
        "SVR"
    ],
    "MAE": [
        gb_mae,
        xgb_mae,
        svr_mae
    ],
    "RMSE": [
        gb_rmse,
        xgb_rmse,
        svr_rmse
    ],
    "R2": [
        gb_r2,
        xgb_r2,
        svr_r2
    ]
})

results

,Model,MAE,RMSE,R2
0,Gradient Boosting,0.198942,0.270768,0.892337
1,XGBoost,0.140217,0.187678,0.948275
2,SVR,0.173188,0.228249,0.923495


In [29]:
import os

os.makedirs("../results", exist_ok=True)

results.to_csv(
    "../results/sam_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!
